In [42]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "src" / "database").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Не найдена папка src/database")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from sqlalchemy import inspect, text
from IPython.display import display
from database.connection import engine

inspector = inspect(engine)
table_names = inspector.get_table_names()

print(f"Подключение успешно. Таблиц: {len(table_names)}")
print(table_names)

Подключение успешно. Таблиц: 12
['datasets', 'dataset_classes', 'annotations', 'sequences', 'detection_sets', 'trackers', 'benchmark_results', 'experiments', 'detections', 'evaluation_runs', 'tracking_results', 'evaluation_metrics']


In [43]:
schema_rows = []

for table_name in table_names:
    for column in inspector.get_columns(table_name):
        schema_rows.append({
            "table": table_name,
            "column": column["name"],
            "type": str(column["type"]),
            "nullable": column["nullable"],
            "primary_key": column.get("primary_key", False),
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

,table,column,type,nullable,primary_key
0,datasets,id,INTEGER,False,False
1,datasets,name,VARCHAR(100),False,False
2,datasets,source,VARCHAR(255),False,False
3,datasets,version,VARCHAR(50),True,False
4,datasets,license,VARCHAR(100),True,False
...,...,...,...,...,...
87,tracking_results,class_id,INTEGER,True,False
88,evaluation_metrics,id,INTEGER,False,False
89,evaluation_metrics,evaluation_run_id,INTEGER,False,False
90,evaluation_metrics,metric_name,VARCHAR(50),False,False


In [44]:
foreign_keys = []

for table_name in table_names:
    for fk in inspector.get_foreign_keys(table_name):
        foreign_keys.append({
            "table": table_name,
            "column": ", ".join(fk["constrained_columns"]),
            "references_table": fk["referred_table"],
            "references_column": ", ".join(fk["referred_columns"]),
        })

display(pd.DataFrame(foreign_keys))

,table,column,references_table,references_column
0,dataset_classes,dataset_id,datasets,id
1,annotations,sequence_id,sequences,id
2,sequences,dataset_id,datasets,id
3,detection_sets,sequence_id,sequences,id
4,benchmark_results,tracker_id,trackers,id
5,experiments,dataset_id,datasets,id
6,experiments,tracker_id,trackers,id
7,detections,detection_set_id,detection_sets,id
8,evaluation_runs,experiment_id,experiments,id
9,tracking_results,experiment_id,experiments,id


In [45]:
from IPython.display import Markdown, display

for table_name in table_names:
    display(Markdown(f"### Таблица `{table_name}`"))
    df = pd.read_sql(
        text(f'SELECT * FROM "{table_name}" LIMIT 10'),
        engine,
    )
    display(df)

### Таблица `datasets`

,id,name,source,version,license,format
0,1,MOT20,Hugging Face: Lekim89/MOT20,MOT20,None,Parquet
1,3,MOT17,MOTChallenge: MOT17,MOT17 train,None,Parquet


### Таблица `dataset_classes`

,id,dataset_id,class_id,name


### Таблица `annotations`

,id,sequence_id,frame,track_id,bbox_left,bbox_top,bbox_width,bbox_height,class_id,mark,visibility
0,1,1,1,1,199.0,813.0,140.0,268.0,1,True,0.83643
1,2,1,2,1,201.0,812.0,140.0,268.0,1,True,0.84015
2,3,1,3,1,203.0,812.0,140.0,268.0,1,True,0.84015
3,4,1,4,1,206.0,812.0,140.0,268.0,1,True,0.84015
4,5,1,5,1,208.0,812.0,140.0,268.0,1,True,0.84015
5,6,1,6,1,211.0,812.0,140.0,268.0,1,True,0.84015
6,7,1,7,1,214.0,812.0,140.0,268.0,1,True,0.84015
7,8,1,8,1,217.0,812.0,140.0,268.0,1,True,0.84015
8,9,1,9,1,220.0,812.0,140.0,268.0,1,True,0.84015
9,10,1,10,1,223.0,812.0,140.0,268.0,1,True,0.84015


... (14 outputs left out)

### Таблица `tracking_results`

,id,experiment_id,sequence_id,frame,track_id,bbox_left,bbox_top,bbox_width,bbox_height,confidence,class_id


### Таблица `evaluation_metrics`

,id,evaluation_run_id,metric_name,metric_value


In [46]:
counts = []

for table_name in table_names:
    query = text(f'SELECT COUNT(*) FROM "{table_name}"')
    count = pd.read_sql(query, engine).iloc[0, 0]
    counts.append({
        "table": table_name,
        "rows": count,
    })

counts_df = pd.DataFrame(counts).sort_values(
    "rows", ascending=False
)
display(counts_df)

,table,rows
2,annotations,1541621
8,detections,891359
4,detection_sets,25
3,sequences,11
0,datasets,2
1,dataset_classes,0
6,benchmark_results,0
5,trackers,0
7,experiments,0
9,evaluation_runs,0


In [47]:
df = pd.read_sql(
    text('SELECT * FROM "annotations"'),
    engine,
)
display(df)

,id,sequence_id,frame,track_id,bbox_left,bbox_top,bbox_width,bbox_height,class_id,mark,visibility
0,1,1,1,1,199.0,813.0,140.0,268.0,1,True,0.836430
1,2,1,2,1,201.0,812.0,140.0,268.0,1,True,0.840150
2,3,1,3,1,203.0,812.0,140.0,268.0,1,True,0.840150
3,4,1,4,1,206.0,812.0,140.0,268.0,1,True,0.840150
4,5,1,5,1,208.0,812.0,140.0,268.0,1,True,0.840150
...,...,...,...,...,...,...,...,...,...,...,...
1541616,1541617,18,428,180,572.0,1044.0,83.0,186.0,10,False,0.197860
1541617,1541618,18,429,180,534.0,1057.0,83.0,186.0,10,False,0.039725
1541618,1541619,18,750,186,1790.0,-151.0,601.0,861.0,10,False,0.172870
1541619,1541620,18,750,187,248.0,588.0,36.0,86.0,12,False,1.000000


In [48]:
queries = {
    "annotations": """
        SELECT d.name AS dataset, a.class_id, COUNT(*) AS rows
        FROM annotations a
        JOIN sequences s ON s.id = a.sequence_id
        JOIN datasets d ON d.id = s.dataset_id
        GROUP BY d.name, a.class_id
        ORDER BY d.name, a.class_id
    """,
    "detections": """
        SELECT d.name AS dataset, det.class_id, COUNT(*) AS rows
        FROM detections det
        JOIN detection_sets ds ON ds.id = det.detection_set_id
        JOIN sequences s ON s.id = ds.sequence_id
        JOIN datasets d ON d.id = s.dataset_id
        GROUP BY d.name, det.class_id
        ORDER BY d.name, det.class_id
    """,
}

for table_name, query in queries.items():
    display(Markdown(f"### `{table_name}`"))
    df = pd.read_sql(text(query), engine)
    display(df)

### `annotations`

,dataset,class_id,rows
0,MOT17,1,112297
1,MOT17,2,1864
2,MOT17,3,6189
3,MOT17,4,13527
4,MOT17,5,1050
5,MOT17,6,11
6,MOT17,7,12041
7,MOT17,8,3595
8,MOT17,9,3724
9,MOT17,10,28573


### `detections`

,dataset,class_id,rows
0,MOT17,None,230216
1,MOT20,None,661143


In [49]:
query = text("""
    SELECT
        d.name AS dataset,
        a.class_id,
        a.mark,
        COUNT(*) AS rows
    FROM annotations a
    JOIN sequences s ON s.id = a.sequence_id
    JOIN datasets d ON d.id = s.dataset_id
    GROUP BY d.name, a.class_id, a.mark
    ORDER BY d.name, a.class_id, a.mark
""")

display(pd.read_sql(query, engine))

,dataset,class_id,mark,rows
0,MOT17,1,True,112297
1,MOT17,2,False,1864
2,MOT17,3,False,6189
3,MOT17,4,False,13527
4,MOT17,5,False,1050
5,MOT17,6,False,11
6,MOT17,7,False,12041
7,MOT17,8,False,3595
8,MOT17,9,False,3724
9,MOT17,10,False,28573


In [50]:
for table_name in table_names:
    print(f"\n### {table_name}")
    for column in inspector.get_columns(table_name):
        print(f"  {column['name']}: {column['type']}")
        


### datasets
  id: INTEGER
  name: VARCHAR(100)
  source: VARCHAR(255)
  version: VARCHAR(50)
  license: VARCHAR(100)
  format: VARCHAR(30)

### dataset_classes
  id: INTEGER
  dataset_id: INTEGER
  class_id: INTEGER
  name: VARCHAR(100)

### annotations
  id: INTEGER
  sequence_id: INTEGER
  frame: INTEGER
  track_id: INTEGER
  bbox_left: DOUBLE PRECISION
  bbox_top: DOUBLE PRECISION
  bbox_width: DOUBLE PRECISION
  bbox_height: DOUBLE PRECISION
  class_id: INTEGER
  mark: BOOLEAN
  visibility: DOUBLE PRECISION

### sequences
  id: INTEGER
  dataset_id: INTEGER
  name: VARCHAR(100)
  fps: DOUBLE PRECISION
  frame_count: INTEGER
  width: INTEGER
  height: INTEGER
  description: VARCHAR(500)
  image_url: VARCHAR(1000)

### detection_sets
  id: INTEGER
  sequence_id: INTEGER
  source: VARCHAR(100)
  detector_name: VARCHAR(100)
  created_at: TIMESTAMP

### trackers
  id: INTEGER
  name: VARCHAR(100)
  family: VARCHAR(100)
  version: VARCHAR(50)
  description: VARCHAR
  repository_url: VA

In [51]:
query = text("""
    SELECT
        d.name AS dataset,
        a.class_id,
        COUNT(*) AS rows
    FROM annotations AS a
    JOIN datasets AS d ON d.id = a.dataset_id
    GROUP BY d.name, a.class_id
    ORDER BY d.name, a.class_id
""")

class_distribution = pd.read_sql(query, engine)
display(class_distribution)

DatabaseError: Execution failed on sql '
    SELECT
        d.name AS dataset,
        a.class_id,
        COUNT(*) AS rows
    FROM annotations AS a
    JOIN datasets AS d ON d.id = a.dataset_id
    GROUP BY d.name, a.class_id
    ORDER BY d.name, a.class_id
': (psycopg.errors.UndefinedColumn) column a.dataset_id does not exist
LINE 7:     JOIN datasets AS d ON d.id = a.dataset_id
                                         ^
[SQL: 
    SELECT
        d.name AS dataset,
        a.class_id,
        COUNT(*) AS rows
    FROM annotations AS a
    JOIN datasets AS d ON d.id = a.dataset_id
    GROUP BY d.name, a.class_id
    ORDER BY d.name, a.class_id
]
(Background on this error at: https://sqlalche.me/e/21/f405)